# Session 5: Give it hands

**Week 3 · Session 1 · Project version v2**

Session 4 ended with an order whose record contradicted the ticket, and an assistant that had no way
to find out. v1 can classify a ticket, copy facts out of it and write a polite reply, but it cannot
check anything. Tonight it gets **tools**: functions it can ask your code to run against ShopWise's
records. By the end it answers questions like *"what did I order last, and when will it arrive?"* by
looking the answer up, four lookups in a row, without being told which.

> *The model never executes anything. It emits a request; your code decides whether to honour it.*

<img src="figures/session-map.png" width="900"
     alt="Session 5 on one page. A top strip contrasts v1, which wrote a polite apology and checked
     nothing, with v2, which made four lookups and gave a grounded answer. Part A, the loop: the
     model asks, your code runs the tool, appends the result and asks again, one more model call than
     there are lookups. Part B, tool design: name, typed arguments, docstring and return, measured on
     fifteen questions, and a tool with a gutted name that the agent silently stops calling. Part C,
     failures: return what the model can fix, raise what it cannot, and keep anything that moves
     money away from it. A band underneath: the model never executes anything; it asks, and your code
     decides.">

Two ideas tonight. **The agent loop**: you write it by hand on a raw model, then meet `create_agent`
and see that it's the same loop. **Designing tools for a model**: what makes the model pick the right
tool, what a tool should hand back, and what to do when it fails. Both are measured, not asserted.

| Section | What it covers |
|---|---|
| 1 | A brain with no hands |
| 2 | Meet the backend |
| 3 | The anatomy of a tool |
| 4 | The loop, by hand |
| 5 | `create_agent` is that loop |
| 6 | Reading the transcript |
| 7 | Designing tools for a model |
| 8 | When a tool fails |
| 9 | A typed object at the end |
| 10 | Project v2 |
| 11 | Wrap-up, and Tanvir's follow-up |

## Setup

Nothing new to install and no new keys. Everything runs on the cheapest model, `settings.MODEL`:
picking one tool from six descriptions is a matching task, not a reasoning one. Anthropic and OpenAI
keys are optional (one cell in section 4.6 uses them; its output is saved).

In [1]:
import json
import logging
import os
import subprocess
import sys
import time
from pathlib import Path

import yaml
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain.messages import AIMessage, HumanMessage, SystemMessage
from langchain.tools import tool

sys.path.insert(0, str(Path.cwd() / "project"))      # the v2 snapshot beside this notebook
from config import settings                            # noqa: E402
from utils import load_inbox                           # noqa: E402
import triage as v1                                    # noqa: E402
import tools as shop                                   # noqa: E402
from agent import MAX_STEPS, answer, transcript        # noqa: E402

# Gemini's SDK logs "there are non-text parts in the response" whenever a reply holds a
# tool request instead of text. It's accurate (section 4.2 explains it) and very noisy.
logging.getLogger("google_genai.types").setLevel(logging.ERROR)

model = init_chat_model(settings.model_id)
by_id = {t.id: t for t in load_inbox()}
print(f"{settings.APP_NAME} v{settings.VERSION} · {settings.model_id}")

ShopWise Support Assistant v2.0.0 · google_genai:gemini-3.5-flash-lite


## 1. A brain with no hands

`TCK-004` is from Tanvir, in Bangla: he ordered a PulseFit smartwatch on the 28th (`ORD-5003`), the
courier says it's in transit, and he wants to know how much longer. Here is v1 on his ticket.

In [2]:
ticket = by_id["TCK-004"]
result, _ = v1.triage(model, ticket)
print(ticket.body, "\n")
print(result.category, "|", result.urgency, "|", result.sentiment)
print(v1.draft_reply(model, ticket, result))

আসসালামু আলাইকুম। আমি ২৮ তারিখে PulseFit স্মার্টওয়াচ অর্ডার করেছি, ORD-5003. এখনো পাইনি। কুরিয়ার বলছে ট্রানজিটে আছে। আর কত দিন লাগবে জানালে ভালো হয়। 

order_status | medium | calm


ওয়ালাইকুম আসসালাম। আপনার PulseFit স্মার্টওয়াচটি এখনও হাতে পাননি জেনে আমি দুঃখিত। নির্ধারিত সময়ের মধ্যে অর্ডার না পেলে অপেক্ষা করা বেশ বিরক্তির বিষয়।

আমার কাছে সরাসরি ডেলিভারি বা কুরিয়ারের কোনো তথ্য নেই, তাই আমি আপনার এই মেসেজটি আমাদের ডেলিভারি টিমের একজন সহকর্মীর কাছে পাঠিয়ে দিচ্ছি। তারা বিষয়টি যাচাই করে এই ইমেইলেই আপনাকে পরবর্তী আপডেট জানিয়ে দেবেন।

ShopWise Support


A good reply: in his language, sympathetic, honest that it can't see the courier, and clear about who
follows up. And not one fact was checked, because there is nothing to check with. Meanwhile the
answer sits in ShopWise's records, one dictionary lookup away:

In [3]:
backend = yaml.safe_load((settings.DATA_DIR / "backend.yml").read_text(encoding="utf-8"))
print("ORD-5003 ", backend["orders"]["ORD-5003"])
print("shipment ", backend["shipments"]["ORD-5003"])

ORD-5003  {'customer_id': 'C-1002', 'items': [{'sku': 'SKU-200', 'qty': 1}], 'total': 199.0, 'status': 'shipped', 'order_date': '2026-07-28', 'delivered_date': None}
shipment  {'carrier': 'SwiftEx', 'tracking_id': 'SW-88231', 'status': 'in_transit', 'eta': '2026-08-08'}


### 1.1 Take the constraints out

v1's prompt has a section that says *"You have no access to orders, payments, shipments or accounts,
so never say you have checked…"*. That section is why the reply was honest. Remove it and ask a
question that can't be politely deflected:

In [4]:
prompt = v1.PROMPTS["system_prompt"]
start = prompt.index("## Constraints")
end = prompt.index("## When you cannot help")
without_constraints = prompt[:start] + prompt[end:]

question = "Just the courier and the tracking number for ORD-5003, please."
for run in range(1, 4):
    reply = model.invoke([("system", without_constraints), ("human", question)])
    print(f"run {run}:", reply.text.replace("<reply>", "").replace("\n", " ").strip()[:150])

print("\ntruth:  ", backend["shipments"]["ORD-5003"]["carrier"], backend["shipments"]["ORD-5003"]["tracking_id"])

run 1: Hello,  Your order ORD-5003 was shipped with Steadfast Courier today. Your tracking number is STF-892341.   You can use this number on their website t


run 2: Hello,  Your order ORD-5003 was shipped with Steadfast Courier today. Your tracking number is STF-884219.   You can use this number on their website t


run 3: Hello,  Your order ORD-5003 has been shipped through our courier partner, Steadfast Courier. Your tracking number is STF-98745231.  You can enter this

truth:   SwiftEx SW-88231


Three runs, three answers that look like data. The courier comes back the same each time, a real
Bangladeshi courier picked because the prompt says the shop is in Bangladesh; the tracking numbers
differ. Both are wrong.

A **stable, confident, wrong** answer is worse than a flaky one. A flaky answer looks like a bug and
gets investigated; a stable one looks like a record and gets forwarded to the customer. The
constraints section was never a fix. It was a bandage over a missing capability, and the capability is
access to the records, under your control.

## 2. Meet the backend

ShopWise's records are five tables in `data/backend.yml`: customers, products, orders,
shipments, refunds. `project/shopwise_data.py` loads them into five dictionaries and does nothing
else. In a real system that file is where you'd open a database connection or call an API; here it's
a YAML file, so it runs offline and costs nothing.

In [5]:
from shopwise_data import CUSTOMERS, ORDERS, PRODUCTS, REFUNDS, SHIPMENTS

for name, table in [("CUSTOMERS", CUSTOMERS), ("PRODUCTS", PRODUCTS), ("ORDERS", ORDERS),
                    ("SHIPMENTS", SHIPMENTS), ("REFUNDS", REFUNDS)]:
    print(f"{name:10} {len(table)} rows")

print("\ntanvir@example.com ->", CUSTOMERS["tanvir@example.com"])
print("ORD-5003           ->", ORDERS["ORD-5003"])

CUSTOMERS  4 rows
PRODUCTS   4 rows
ORDERS     4 rows
SHIPMENTS  1 rows
REFUNDS    1 rows

tanvir@example.com -> {'customer_id': 'C-1002', 'name': 'Tanvir Ahmed', 'tier': 'regular', 'since': '2024-07-02'}
ORD-5003           -> {'customer_id': 'C-1002', 'items': [{'sku': 'SKU-200', 'qty': 1}], 'total': 199.0, 'status': 'shipped', 'order_date': '2026-07-28', 'delivered_date': None}


Each table is keyed by what a customer actually types: customers by email, orders by order ID,
shipments by **order** ID (nobody asks "where is SW-88231?"). A few records are deliberate, and
tonight's demos depend on them:

| Record | Why it's there |
|---|---|
| `ORD-5003` (Tanvir) | the only order that has shipped, so the only one with tracking |
| `kamal@example.com` (C-1004) | a customer with **no orders**: an empty list is an answer, not an error |
| `SKU-300`, the kettle | `stock: 0`, an honest "no" |
| `SKU-400`, the backpack | `warranty_months: 0`, no warranty at all |
| `REF-9001` | status `requested`: logged, no money moved yet |
| `ORD-5004` | delivered in May, far outside the 30-day return window (session 9) |

The data lives once, beside `tickets.yml`, and every snapshot from v2 reads it through the same
five-line loader rather than copying it (ADR-0006).

## 3. The anatomy of a tool

A **tool** is a function the model may *ask* your code to run. `@tool` turns a plain Python function
into one. The model never sees the code. It sees four things, and decides from those alone whether
this tool helps and what to pass it.

<img src="figures/tool-anatomy.png" width="900"
     alt="The anatomy of a tool. On the left, a Python function decorated with @tool, with four numbered
     callouts: one, the name, a verb such as get_shipment; two, the typed argument order_id of type
     str; three, the docstring, a summary line saying what it does and what comes back, a line saying
     when to use it, and an Args line per argument; four, the return, plain data. On the right, the
     JSON the model actually receives: name, description, and parameters with order_id as a required
     string carrying its own description. Arrows connect each part to where it lands. A trap panel
     notes that a Returns section is dropped and the function body is never sent. A band underneath:
     the model reads the description, never the body.">

| Part | What it does for the model |
|---|---|
| **Name** | a verb on a noun: `get_shipment`. The first thing a question is matched against |
| **Typed arguments** | the type hints become a JSON schema the model must fill in. No hint, no type |
| **Docstring** | one line saying what it does and what comes back, a line saying when to use it, and one line per argument. It's how the model picks one tool out of several |
| **Return** | plain data (a dict or list) that survives being turned into text, because text is what the model reads |

**Docs:** [tools](https://docs.langchain.com/oss/python/langchain/tools) ·
[creating tools](https://docs.langchain.com/oss/python/langchain/tools#create-tools) ·
[Gemini function calling](https://ai.google.dev/gemini-api/docs/function-calling)

Here is one of ShopWise's tools, and exactly what the model receives from it:

In [6]:
from langchain_core.utils.function_calling import convert_to_openai_tool

print(type(shop.get_shipment).__name__, "|", shop.get_shipment.name, "\n")
print(json.dumps(convert_to_openai_tool(shop.get_shipment)["function"], indent=2))

StructuredTool | get_shipment 

{
  "name": "get_shipment",
  "description": "Track a shipped order: carrier, tracking ID, delivery status and estimated arrival date. Use for \"where is my parcel?\" questions. Returns {\"error\": ...} if there is no\ntracking for that order, which is a normal answer for an order that hasn't shipped.",
  "parameters": {
    "properties": {
      "order_id": {
        "description": "An order ID like \"ORD-5003\".",
        "type": "string"
      }
    },
    "required": [
      "order_id"
    ],
    "type": "object"
  }
}


`convert_to_openai_tool` shows the definition in the common wire format; each vendor's integration
sends its own equivalent. The function body, the `.strip().upper()`, the error handling: none of it is
there. If the description is vague, the model is guessing.

### 3.1 Where each line of the docstring goes

ShopWise's tools use `@tool(parse_docstring=True)`. It reads a Google-style docstring and moves each
`Args:` line into the schema, next to the argument it describes (that's the `description` under
`order_id` above). One consequence nobody tells you:

In [7]:
@tool(parse_docstring=True)
def get_refund(refund_id: str) -> dict:
    """Check a refund the customer has already requested.

    Args:
        refund_id: A refund ID like "REF-9001".

    Returns:
        Status, amount and reason. Status "requested" means no money has moved yet.
    """
    return {}

print(repr(get_refund.description))

'Check a refund the customer has already requested.'


The `Returns:` section is gone, silently. The model never learns that `"requested"` means no money has
moved. That's why every ShopWise tool says what comes back in its first line ("…: carrier, tracking
ID, status and estimated arrival date") rather than under `Returns:`. Without `parse_docstring`, the
whole docstring is sent as one block of text, `Args:` and all, which also works but gives the
arguments no descriptions of their own.

### 3.2 It's still an ordinary function

A tool is a Runnable: call it with `.invoke` and a **dict** of arguments, no model involved. Hand it a
whole tool call (the shape the model produces, section 4) and you get back a `ToolMessage`, ready to
append to a conversation.

In [8]:
print(shop.get_shipment.invoke({"order_id": "ORD-5003"}))

call = {"name": "get_shipment", "args": {"order_id": "ORD-5003"}, "id": "call_1", "type": "tool_call"}
print(repr(shop.get_shipment.invoke(call)))

{'order_id': 'ORD-5003', 'carrier': 'SwiftEx', 'tracking_id': 'SW-88231', 'status': 'in_transit', 'eta': '2026-08-08'}
ToolMessage(content='{"order_id": "ORD-5003", "carrier": "SwiftEx", "tracking_id": "SW-88231", "status": "in_transit", "eta": "2026-08-08"}', name='get_shipment', tool_call_id='call_1')


Test a tool like this before any model sees it. `uv run python tools.py` in the project does exactly
that for all seven: no key, no model, and if something's wrong the traceback is yours.

### 3.3 Two argument names you must not use

`config` and `runtime` are **reserved**: LangChain fills them in itself. Use either as an ordinary
argument and the failure is strange:

In [9]:
@tool
def get_order_bad(order_id: str, config: dict) -> dict:
    """Get one order by its ID."""
    return {"order_id": order_id}

print("the model is shown:", list(get_order_bad.args))
try:
    get_order_bad.invoke({"order_id": "ORD-5003", "config": {"verbose": True}})
except TypeError as err:
    print("TypeError:", err)

the model is shown: ['order_id', 'config']
TypeError: get_order_bad() missing 1 required positional argument: 'config'


The model is *shown* `config`, fills it in, LangChain strips it because the name is reserved, and
your function fails with a missing argument you can see in the schema. Nothing warns you when you
define it. The supported way is a parameter annotated `ToolRuntime`, which is hidden from the model
and filled in by the framework:

In [10]:
from langchain.tools import ToolRuntime


@tool
def get_order_good(order_id: str, runtime: ToolRuntime) -> dict:
    """Get one order by its ID."""
    return {"order_id": order_id, "asked_in_call": runtime.tool_call_id}

print("the model is shown:", list(get_order_good.args))

the model is shown: ['order_id']


`runtime` carries the conversation's state, its configuration and, from session 11, facts your
application supplies per request, such as *who is logged in*. Older tutorials use `InjectedState`,
`InjectedStore` or `get_runtime()`; the docs now say to use `ToolRuntime` instead.
**Docs:** [reserved argument names](https://docs.langchain.com/oss/python/langchain/tools#reserved-argument-names) ·
[accessing context](https://docs.langchain.com/oss/python/langchain/tools#access-context)

### 3.4 Seven tools, six given

`project/tools.py` defines the seven tools whose signatures are fixed for the rest of the course:

| Tool | Reads or writes | Answers |
|---|---|---|
| `lookup_customer(email)` | reads | who is this? returns the `customer_id` the next tool needs |
| `get_order(order_id)` | reads | status, dates, total, items **with product names** |
| `list_orders(customer_id)` | reads | every order, newest first; `[]` for none |
| `get_shipment(order_id)` | reads | carrier, tracking ID, status, ETA |
| `get_product(sku_or_name)` | reads | price, stock, warranty; matches "kettle" as well as `SKU-300` |
| `check_refund_status(refund_id)` | reads | where a refund has got to |
| `issue_refund(order_id, amount, reason)` | **moves money** | creates a refund |

The seventh works. Watch it:

In [11]:
before = len(REFUNDS)
created = shop.issue_refund.invoke({"order_id": "ORD-5003", "amount": 199.0, "reason": "demo"})
print(created)
print("refunds on record:", before, "->", len(REFUNDS))

del REFUNDS[created["refund_id"]]            # undo it, so later cells see the fixture as it is
print("given to the agent:", [t.name for t in shop.TOOLS])

{'refund_id': 'REF-9002', 'order_id': 'ORD-5003', 'amount': 199.0, 'status': 'processed', 'reason': 'demo', 'requested_date': '2026-07-28'}
refunds on record: 1 -> 2
given to the agent: ['lookup_customer', 'get_order', 'list_orders', 'get_shipment', 'get_product', 'check_refund_status']


It moved money, and it isn't in `TOOLS`. Six tools read; one writes, and nothing in v2 decides whether
a refund is deserved, so the model doesn't get to either. This is **least privilege**, and OWASP lists
its absence ("excessive agency": too many tools, too much permission, too much autonomy) as one of the
top ten risks of LLM applications. Session 7 gives the agent `issue_refund` with a person approving
every call. **Docs:** [OWASP LLM06: Excessive Agency](https://genai.owasp.org/llmrisk/llm062025-excessive-agency/)

## 4. The loop, by hand

This is the section to understand completely. Everything after it, `create_agent` included, is this
loop with conveniences added.

<img src="figures/loop-by-hand.png" width="900"
     alt="The agent loop written by hand. Five numbered steps run clockwise around a central message list:
     one, bind the tools to the model; two, invoke it, and an AI message comes back holding a tool
     request, with empty content; three, your code runs the tool and builds a tool message carrying
     the same call ID; four, append both messages; five, invoke again, until a reply comes back with
     no tool calls. The message list in the middle grows: human, AI asking for get_shipment, tool
     result, AI answer. A side note shows input tokens climbing with every call, because the whole
     list is sent each time.">

**Docs:** [tool calling](https://docs.langchain.com/oss/python/langchain/models#tool-calling) ·
[the tool execution loop](https://docs.langchain.com/oss/python/langchain/models#tool-execution-loop)

### 4.1 Bind the tools, ask a question

`bind_tools` returns a copy of the model that sends the six tool definitions with every call.

In [12]:
model_with_tools = model.bind_tools(shop.TOOLS)

question = "Where is my order ORD-5003, and when will it arrive?"
ai = model_with_tools.invoke(question)

print("content:   ", repr(ai.content))
print("text:      ", repr(ai.text))
print("tool_calls:", ai.tool_calls)

content:    []
text:       ''
tool_calls: [{'name': 'get_shipment', 'args': {'order_id': 'ORD-5003'}, 'id': 'call_2192096', 'type': 'tool_call'}]


### 4.2 What came back is not an answer

`.content` is an empty list and `.text` is an empty string, with no warning. If you only print
`.text`, your loop looks broken. The reply is in `.tool_calls`: a **request** to run `get_shipment`
with `order_id="ORD-5003"`, plus an `id` that the result must quote back. Nothing has been executed.
The model wrote a name and some arguments where its text would have been.

Check `ai.tool_calls` before you read `ai.text`. That one `if` is what separates "the model asked for
something" from "the model answered".

Two more things travelled with that request:

In [13]:
print("additional_kwargs:", list(ai.additional_kwargs))

with_tools = ai.usage_metadata["input_tokens"]
without_tools = model.invoke(question).usage_metadata["input_tokens"]
print(f"input tokens, same question: {without_tools} without tools, {with_tools} with six")

additional_kwargs: ['function_call', '__gemini_function_call_thought_signatures__']


input tokens, same question: 18 without tools, 701 with six


- **The tool definitions are billed on every call**, used or not. Six tools cost several hundred input
  tokens before the question is even read. A tool set is a recurring cost, which is one reason to keep
  it small.
- `__gemini_function_call_thought_signatures__` is Gemini 3's **thought signature**: an encrypted
  record of the reasoning behind the request. Google requires it to be sent back with the result. It
  only goes back if you append the message you received, so never rebuild it. Section 4.6 shows why.

### 4.3 Run it: you, not the model

In [14]:
tools_by_name = {t.name: t for t in shop.TOOLS}

results = []
for call in ai.tool_calls:
    tool_message = tools_by_name[call["name"]].invoke(call)     # the whole call in, a ToolMessage out
    results.append(tool_message)
    print(tool_message.tool_call_id, "->", tool_message.content)

call_2192096 -> {"order_id": "ORD-5003", "carrier": "SwiftEx", "tracking_id": "SW-88231", "status": "in_transit", "eta": "2026-08-08"}


This loop is the moment the request becomes an action, and it's *your* code. It's also where you could
decide **not** to run it: check the arguments, ask a person, refuse. Session 7 puts exactly that
here.

### 4.4 Append both, ask again

In [15]:
messages = [HumanMessage(question), ai, *results]
final = model_with_tools.invoke(messages)

print("tool_calls:", final.tool_calls)
print(final.text)

tool_calls: []
Order **ORD-5003** is currently in transit with **SwiftEx** (tracking number: `SW-88231`), and it is estimated to arrive on **August 8, 2026**.


No tool calls this time, so this is the answer, built from the result your code handed back.

### 4.5 Read the conversation

In [16]:
for number, message in enumerate([*messages, final], start=1):
    if message.type == "ai" and message.tool_calls:
        what = "asks for " + ", ".join(c["name"] + str(c["args"]) for c in message.tool_calls)
    else:
        what = message.text.replace("\n", " ")[:90]
    print(f"{number}  {type(message).__name__:12} {what}")

1  HumanMessage Where is my order ORD-5003, and when will it arrive?
2  AIMessage    asks for get_shipment{'order_id': 'ORD-5003'}
3  ToolMessage  {"order_id": "ORD-5003", "carrier": "SwiftEx", "tracking_id": "SW-88231", "status": "in_tr
4  AIMessage    Order **ORD-5003** is currently in transit with **SwiftEx** (tracking number: `SW-88231`),


| # | Written by | |
|---|---|---|
| 1 | the customer | the question |
| 2 | **the model** | a request, not an action |
| 3 | **your code** | the only message containing a fact from ShopWise's records |
| 4 | the model | the answer, written from message 3 |

The model is **stateless**: call 4 knew about the shipment only because messages 1 to 3 were sent
again. The whole memory of this conversation is a Python list called `messages`. Hold on to that for
session 6.

### 4.6 Two rules of the protocol

The message list isn't free-form. It's a protocol, and vendors enforce it differently.

**Answer every request.** A model can ask for several tools in one message (*parallel* tool calls)
when the lookups don't depend on each other:

In [17]:
two = "What's the status of ORD-5002, and of ORD-5003?"
ai = model_with_tools.invoke(two)
print(len(ai.tool_calls), "requests in one message:", [(c["name"], c["args"]) for c in ai.tool_calls])

# Answer only the first one and ask again.
first_only = [HumanMessage(two), ai, tools_by_name[ai.tool_calls[0]["name"]].invoke(ai.tool_calls[0])]
again = model_with_tools.invoke(first_only)
print("Gemini, one request unanswered ->", again.tool_calls or again.text)

2 requests in one message: [('get_order', {'order_id': 'ORD-5002'}), ('get_order', {'order_id': 'ORD-5003'})]


Gemini, one request unanswered -> [{'name': 'get_order', 'args': {'order_id': 'ORD-5003'}, 'id': 'call_710039', 'type': 'tool_call'}]


Gemini lets it go and asks again for the request you skipped: one extra call, billed. Other vendors
refuse the whole conversation:

In [18]:
for model_id, key_name in [("anthropic:claude-haiku-4-5-20251001", "ANTHROPIC_API_KEY"),
                           ("openai:gpt-5.4-nano", "OPENAI_API_KEY")]:
    if not os.getenv(key_name):
        print(f"{model_id}: skipped, no {key_name}")
        continue
    other = init_chat_model(model_id).bind_tools(shop.TOOLS)
    asked = other.invoke(two)
    first_only = [HumanMessage(two), asked,
                  tools_by_name[asked.tool_calls[0]["name"]].invoke(asked.tool_calls[0])]
    try:
        other.invoke(first_only)
        print(f"{model_id}: accepted")
    except Exception as err:
        print(f"{model_id}: {type(err).__name__}: {str(err)[:150]}")

anthropic:claude-haiku-4-5-20251001: AnthropicInvalidRequestError: Error code: 400 - {'type': 'error', 'error': {'type': 'invalid_request_error', 'message': 'messages.2: `tool_use` ids were found without `tool_result`


openai:gpt-5.4-nano: OpenAIInvalidRequestError: Error code: 400 - {'error': {'message': "An assistant message with 'tool_calls' must be followed by tool messages responding to each 'tool_call_id'. T


v1's fallback can be a different vendor (session 4), so your loop must obey the strictest one: every
request in the message gets a `ToolMessage` with its `tool_call_id`, in any order. **Docs:**
[parallel tool calls](https://docs.langchain.com/oss/python/langchain/models#parallel-tool-calls)

**Append the message you received.** It's tempting to tidy up and append
`AIMessage(content="", tool_calls=ai.tool_calls)` instead of `ai`. On Gemini 3 that drops the thought
signature. Google requires it in a tool loop, so the integration quietly inserts a placeholder
signature to keep the request valid. The call succeeds, and the model has lost the reasoning behind its
own request. Append `ai` itself.
**Docs:** [thought signatures](https://ai.google.dev/gemini-api/docs/thinking) ·
[the LangChain Gemini integration](https://docs.langchain.com/oss/python/integrations/chat/google_generative_ai)

### 4.7 Put a loop round it, with a budget

Steps 2 to 5, repeated until the model stops asking. The `max_calls` budget isn't optional: a loop
whose exit depends on what a model decides to say must have a second way out.

In [19]:
def run_loop(question, max_calls=6):
    """Section 4's steps in a loop, with a budget. Returns the message list."""
    messages = [SystemMessage(v1.PROMPTS["agent_prompt"]), HumanMessage(question)]
    model_seconds = tool_seconds = 0.0

    for number in range(1, max_calls + 1):
        start = time.perf_counter()
        ai = model_with_tools.invoke(messages)
        model_seconds += time.perf_counter() - start
        messages.append(ai)
        asked = [c["name"] for c in ai.tool_calls]
        print(f"call {number}: {ai.usage_metadata['input_tokens']:5} input tokens   asks {asked or 'nothing'}")
        if not ai.tool_calls:
            break

        start = time.perf_counter()
        for call in ai.tool_calls:
            messages.append(tools_by_name[call["name"]].invoke(call))
        tool_seconds += time.perf_counter() - start

    if messages[-1].type != "ai" or messages[-1].tool_calls:
        print(f"stopped by the budget: {max_calls} calls and still asking")
    print(f"\ntime in the model {model_seconds:.2f} s, in the tools {tool_seconds * 1000:.1f} ms")
    print("answer:", messages[-1].text)
    return messages


loop_messages = run_loop("Hi, I'm tanvir@example.com. What did I order most recently, and when will it arrive?")

call 1:  1013 input tokens   asks ['lookup_customer']


call 2:  1095 input tokens   asks ['list_orders']


call 3:  1172 input tokens   asks ['get_shipment']


call 4:  1262 input tokens   asks ['get_order']


call 5:  1384 input tokens   asks nothing

time in the model 4.00 s, in the tools 7.8 ms
answer: Your most recent order was the PulseFit Smartwatch, placed on July 28, 2026. It has shipped via SwiftEx and is expected to arrive by August 8, 2026.


Nobody told it the route. It turned an email into a customer ID, the ID into a list of orders, and
the latest order into a shipment, each step using a value the previous result supplied. Three things
to read off that output:

- **Model calls = lookups + 1.** Somebody has to write the answer after the last lookup.
- **Input tokens climb on every call**, because the whole list is resent each time, tool definitions
  included. Long agent runs get expensive in input, not output.
- **Almost all the time is the model.** The tools are dictionary lookups and take milliseconds. With a
  real database the tool column grows, so measure yours; either way the customer waits for the whole
  chain with nothing on screen, which is session 6's streaming.

## 5. `create_agent` is that loop

<img src="figures/loop-boxed.png" width="900"
     alt="The same five-step loop as the previous figure, drawn in the same positions, with one change: a
     large rectangle surrounds the whole loop, labelled create_agent with the model, the tools and
     the system prompt. A small gauge on the rectangle's edge is labelled recursion_limit, the
     budget, with a note that create_agent allows 9,999 steps unless you set it. A band underneath:
     create_agent is that loop, plus a budget you have to set.">

Same five steps, same message list, with a box drawn round them. `create_agent` takes a model, a list
of tools and a system prompt, and runs section 4.7's loop for you.

**Docs:** [agents](https://docs.langchain.com/oss/python/langchain/agents) ·
[component architecture](https://docs.langchain.com/oss/python/langchain/component-architecture)

The system prompt is new in v2, in `prompts.yml` beside v1's. It's v1's constraints section turned
inside out: v1 said *you have no access*; this says *the tools are your only source of facts*.

In [20]:
print(v1.PROMPTS["agent_prompt"])

## Identity
You are the customer support assistant for ShopWise, an online electronics
store in Bangladesh selling audio, wearables and small kitchen appliances.
You answer customers in a chat.

## Instructions
- Your tools read ShopWise's records. They are your only source of facts
  about customers, orders, products, shipments and refunds.
- Look up every fact before you state it, and chain as many lookups as the
  question needs.
- When a customer gives an email, call lookup_customer first: list_orders
  needs the customer_id it returns.
- Reply in the language the customer wrote in.

## Constraints, and why they exist
- Never state a status, date, carrier, tracking number, price, stock level
  or refund amount that no tool returned in this conversation. A wrong fact
  sent to a customer costs ShopWise more than "I couldn't find that".
- Only a colleague can approve refunds, changes to an order or delivery
  promises, so never promise one. Say who will follow up instead.

## When a 

In [21]:
agent = create_agent(model, shop.TOOLS, system_prompt=v1.PROMPTS["agent_prompt"])

result = agent.invoke(
    {"messages": [{"role": "user", "content": "Hi, I'm tanvir@example.com. What did I order most recently, and when will it arrive?"}]},
    config={"recursion_limit": MAX_STEPS},
)
print(transcript(result))

CUSTOMER   Hi, I'm tanvir@example.com. What did I order most recently, and when will it arrive?
  ASKS     lookup_customer({'email': 'tanvir@example.com'})
  RESULT   lookup_customer -> {"email": "tanvir@example.com", "customer_id": "C-1002", "name": "Tanvir Ahmed", "tier": "regular", "since": "2024-07-02"}
  ASKS     list_orders({'customer_id': 'C-1002'})
  RESULT   list_orders -> [{"order_id": "ORD-5003", "status": "shipped", "order_date": "2026-07-28", "total": 199.0}]
  ASKS     get_shipment({'order_id': 'ORD-5003'})
  RESULT   get_shipment -> {"order_id": "ORD-5003", "carrier": "SwiftEx", "tracking_id": "SW-88231", "status": "in_transit", "eta": "2026-08-08"}
  ASKS     get_order({'order_id': 'ORD-5003'})
  RESULT   get_order -> {"order_id": "ORD-5003", "customer_id": "C-1002", "items": [{"sku": "SKU-200", "name": "PulseFit Smartwatch", "qty": 1, "unit_price": 199.0}], "total": 199.0, "status": "shipped", "order_date": "2026-07-28", "delivered_date": null}
ASSISTANT  Your most rec

`result["messages"]` is the same list you built by hand: human, the model's requests, your tools'
results, the answer. `transcript()` in `project/agent.py` just prints it readably. `ASKS` lines are
the model; `RESULT` lines are your code.

### 5.1 The budget is your job

`recursion_limit` counts **steps**: each model call is one, each round of tool execution is another,
so one lookup costs two. `create_agent` sets its own limit, and it isn't a safety net:

In [22]:
print("create_agent's default:", agent.config["recursion_limit"])

try:
    agent.invoke({"messages": [{"role": "user", "content": "Hi, I'm tanvir@example.com. What did I order most recently, and when will it arrive?"}]},
                 config={"recursion_limit": 4})
except Exception as err:
    print(type(err).__name__, ":", str(err).splitlines()[0])

create_agent's default: 9999


GraphRecursionError : Recursion limit of 4 reached without hitting a stop condition. You can increase the limit by setting the `recursion_limit` config key.


9,999 steps is roughly 5,000 billed model calls before anything stops a confused agent. Tutorials
written before LangChain v1 say the limit is 25; it isn't any more. v2 passes `MAX_STEPS = 12` (five
lookups and an answer) on every call. Session 7 replaces it with middleware that counts model and tool
calls directly. **Docs:** [GRAPH_RECURSION_LIMIT](https://docs.langchain.com/oss/python/langgraph/errors/GRAPH_RECURSION_LIMIT)

## 6. Reading the transcript

The reply is the least informative part of an agent run. Two questions with less obvious routes:

In [23]:
for q in ["I'm kamal@example.com. Can you list my orders?",
          "Is the BrewMaster kettle in stock, and what warranty does it have?"]:
    print(transcript(answer(agent, q)), "\n")

CUSTOMER   I'm kamal@example.com. Can you list my orders?
  ASKS     lookup_customer({'email': 'kamal@example.com'})
  RESULT   lookup_customer -> {"email": "kamal@example.com", "customer_id": "C-1004", "name": "Kamal Hossain", "tier": "gold", "since": "2021-05-18"}
  ASKS     list_orders({'customer_id': 'C-1004'})
  RESULT   list_orders -> []
ASSISTANT  I found your account under the name Kamal Hossain, but our records show that you have not placed any orders yet. Let me know if you need help finding a product to buy. 



CUSTOMER   Is the BrewMaster kettle in stock, and what warranty does it have?
  ASKS     get_product({'sku_or_name': 'BrewMaster kettle'})
  RESULT   get_product -> {"error": "no product matches 'BrewMaster kettle'.", "available": [{"sku": "SKU-100", "name": "AeroSound Pro Headphones"}, {"sku": "SKU-200", "name": "PulseFit Smartwatch"}, {"sku": "SKU-300", "name": "BrewMaster Electric Kettle"}, {"sku": "SKU-400", "name": "TrailPack 30L Backpack"}]}
  ASKS     get_product({'sku_or_name': 'BrewMaster Electric Kettle'})
  RESULT   get_product -> {"sku": "SKU-300", "name": "BrewMaster Electric Kettle", "product_type": "home", "price": 49.0, "stock": 0, "warranty_months": 6}
ASSISTANT  The BrewMaster Electric Kettle is currently out of stock. It comes with a six-month warranty when available. 



- **Kamal has no orders.** `list_orders` returned `[]` and the agent said so. An empty list is a
  successful lookup; had the tool returned `{"error": ...}` for it, the agent would have told a real
  customer that something failed.
- **The kettle.** In this run the agent searched for "BrewMaster kettle", which matches nothing (the
  catalogue says "BrewMaster Electric Kettle"). The tool's error listed what *does* exist; the agent
  read it, corrected its own argument and asked again. Nobody wrote that retry. It came from an error
  message written for the model to act on (section 8).
- **The path is the model's decision, not a contract.** Run a cell twice and the order of lookups can
  change. Never write a test that asserts an exact sequence of tool calls; assert the *set* that must
  happen (section 7.1). If you need a fixed sequence, you need a graph (session 10), not an agent.

### 6.1 The transcript is the trace

With `LANGSMITH_TRACING=true` (session 4), every run above is already in LangSmith, named `agent`: the
same messages, plus latency and tokens per step and the tool definitions sent with each call. When an
agent answers something surprising, open its run before you change the prompt. The question to ask is
not "what did it say?" but "what did it look up, and what came back?"

## 7. Designing tools for a model

Teams that build agents for a living report the same thing: the tools matter more than the prompt.
Anthropic, describing their own agent work: *"we actually spent more time optimizing our tools than
the overall prompt."* A tool's name, arguments and docstring are a prompt, read on every call, and a
tool's return value is the only evidence the model gets.

<img src="figures/tool-design.png" width="900"
     alt="Designing tools for a model, as a checklist of seven rules with a pictogram each: name the tool
     with a verb; write the docstring for a new hire, saying what comes back and when to use it;
     return names rather than bare codes; keep the set small and distinct, under twenty; write errors
     that say what would work; never make the model fill in what your code already knows, such as who
     is asking; and separate tools that read from tools that move money. Beside the checklist, a
     measured panel: tool selection fifteen out of fifteen, docstring gutted still four out of four,
     name gutted zero out of four with no error. A band underneath: a tool description is a prompt,
     so measure it.">

| Rule | Why | In ShopWise |
|---|---|---|
| Name it with a verb on a noun | it's matched against the question first | `get_shipment`, not `shipping_handler` |
| Write the docstring for a new hire | the model has no other context | what comes back, when to use it, one line per argument |
| Unambiguous argument names | `id` could be anything | `order_id`, `customer_id`, `refund_id` |
| Return names and meanings, not bare codes | otherwise the model guesses or spends a lookup | `get_order` adds product names to SKUs |
| Keep the set small and distinct | overlapping tools get confused; each definition costs tokens | six tools; OpenAI suggests under 20, Google 10 to 20 |
| Errors that say what would work | an error is the model's only clue | `"Order IDs look like ORD-5001."` |
| Never make the model supply what you already know | the model will fill it in, wrongly or for the wrong person | who is asking comes from the login, not an argument (section 11.4) |
| Separate reading from acting | one is safe to automate, the other isn't | `issue_refund` withheld |

**Docs:** [Anthropic: writing effective tools for agents](https://www.anthropic.com/engineering/writing-tools-for-agents) ·
[OpenAI: function calling](https://developers.openai.com/api/docs/guides/function-calling) ·
[Google: function calling best practices](https://ai.google.dev/gemini-api/docs/function-calling)

### 7.1 Measure tool selection

Session 3's habit: before trusting a change, score it on cases written down beforehand.
`tool_questions.yml` (beside this notebook) holds fifteen questions, each with the tools a correct run
**must** call. Order doesn't matter and extra calls aren't failures, because the route is the model's
choice. One question must call no tool at all. `score_tools.py` runs them:

In [24]:
import score_tools

baseline = score_tools.score(agent, score_tools.load_questions())
print(f"\ntool selection {baseline['passed']}/{baseline['total']} · "
      f"{baseline['model_calls']} model calls · {baseline['input_tokens']:,} input tokens")

  ok    Where is my order ORD-5003?                                 get_shipment


  ok    What's the tracking number for ORD-5003?                    get_shipment


  ok    ORD-5003 অর্ডারটি এখন কোথায় আছে?                           get_shipment


  ok    tanvir@example.com here. When will my latest order arrive?  lookup_customer, list_orders, get_shipment


  ok    What's in order ORD-5002?                                   get_order


...

  ok    How much is the PulseFit smartwatch?                        get_product


  ok    Does the TrailPack backpack come with a warranty?           get_product, get_product


  ok    Has my refund REF-9001 gone through yet?                    check_refund_status


  ok    What's the status of refund REF-9002?                       check_refund_status


  ok    What are your customer service hours?                       -

tool selection 15/15 · 34 model calls · 36,221 input tokens


That's the baseline, including the Bangla question. From now on, any change to a tool's name,
arguments or docstring gets scored against it. The ⭐⭐⭐ exercise adds questions of your own.

### 7.2 Docstring surgery, measured

How much of the selection does each part carry? Rebuild `get_shipment` worse, one part at a time, and
rerun only the four shipment questions:

In [25]:
others = [t for t in shop.TOOLS if t.name != "get_shipment"]


@tool
def get_shipment(order_id: str) -> dict:
    """Gets data."""
    return shop.get_shipment.invoke({"order_id": order_id})


@tool
def helper(x: str) -> dict:
    """Gets data."""
    return shop.get_shipment.invoke({"order_id": x})


shipment_questions = score_tools.load_questions("shipment")
for label, version in [("as shipped", shop.get_shipment), ("docstring gutted", get_shipment),
                       ("name gutted too", helper)]:
    variant = create_agent(model, others + [version], system_prompt=v1.PROMPTS["agent_prompt"])
    print(label)
    s = score_tools.score(variant, shipment_questions)
    print(f"  -> {s['passed']}/{s['total']}\n")

as shipped


  ok    Where is my order ORD-5003?                                 get_shipment


  ok    What's the tracking number for ORD-5003?                    get_shipment


  ok    ORD-5003 অর্ডারটি এখন কোথায় আছে?                           get_shipment


  ok    tanvir@example.com here. When will my latest order arrive?  lookup_customer, list_orders, get_shipment
  -> 4/4

docstring gutted


...

  ok    tanvir@example.com here. When will my latest order arrive?  lookup_customer, list_orders, get_order, get_shipment
  -> 4/4

name gutted too


  FAIL  Where is my order ORD-5003?                                 get_order


  FAIL  What's the tracking number for ORD-5003?                    get_order


  FAIL  ORD-5003 অর্ডারটি এখন কোথায় আছে?                           get_order


  FAIL  tanvir@example.com here. When will my latest order arrive?  lookup_customer, list_orders, get_order, helper
  -> 0/4



- **Gutting the docstring didn't break it.** The name `get_shipment`, the argument `order_id` and the
  question's own words are three signals, and the model needed one. The parts are redundant, which is
  good design, and also why a bad docstring survives code review.
- **Gutting the name broke it completely, and silently.** No error, no apology: the agent fell back to
  `get_order` and answered fluently with whatever that returned, minus the tracking number that was
  one call away. That's what a tool-selection failure looks like in production. You only see it in
  the transcript or a score.

The review comment to leave on `def helper(x): "Gets data."` is not a naming nit: *the model cannot see
the body. Name it for what it does, and say in the docstring what it returns and when to use it.*

### 7.3 Return what the model needs

The stored order holds SKUs only (`{"sku": "SKU-300", "qty": 2}`). ShopWise's `get_order` adds the
product name before returning. Here's a version that doesn't bother:

In [26]:
@tool("get_order", parse_docstring=True)
def get_order_raw(order_id: str) -> dict:
    """Get one order by its ID: status, order and delivery dates, total, and the items in it.

    Use when the customer names an order. Returns {"error": ...} if no order has that ID.

    Args:
        order_id: An order ID like "ORD-5001".
    """
    return ORDERS.get(order_id.strip().upper(), {"error": "no such order"})


not_get_order = [t for t in shop.TOOLS if t.name != "get_order"]
for label, version in [("names added", shop.get_order), ("raw SKUs", get_order_raw)]:
    variant = create_agent(model, not_get_order + [version], system_prompt=v1.PROMPTS["agent_prompt"])
    for run in range(2):
        result = answer(variant, "What's in my order ORD-5002?")
        print(f"{label:12} asks {score_tools.tools_called(result)}  {result['messages'][-1].text[:80]}")

names added  asks ['get_order']  Your order ORD-5002 contains 2 units of the BrewMaster Electric Kettle. The orde


names added  asks ['get_order']  Your order ORD-5002 contains 2 units of the BrewMaster Electric Kettle, with a t


raw SKUs     asks ['get_order', 'get_product']  Your order ORD-5002 contains 2 units of the BrewMaster Electric Kettle, with a t


raw SKUs     asks ['get_order', 'get_product']  Your order ORD-5002 contains 2 units of the BrewMaster Electric Kettle (SKU-300)


With raw SKUs the agent spent an extra `get_product` lookup every time: one more model call on every
order question, forever, for a join your tool could do in one line. The alternative is worse: a model
that skips the lookup has to guess the product's name. Return the fields that help the model answer,
in the words it will repeat to the customer, and leave the rest out.

### 7.4 Who is asking?

Look at `lookup_customer(email)` and `list_orders(customer_id)`: the *model* supplies the identity, from
whatever the customer typed. That's fine for a demo and wrong in production, where the logged-in user
is something your application already knows. Section 11.4 shows what happens when nobody checks, and
session 11 passes identity through `ToolRuntime` instead of trusting an argument.

## 8. When a tool fails

<img src="figures/tool-failure.png" width="900"
     alt="When a tool call does not work, a decision chart with five branches. A fact about the world,
     such as stock zero or an empty order list: return it as data, because the lookup worked. A
     request the model can fix, such as a wrong order ID or an ambiguous product name: return an
     error that says what would work. Bad arguments: the framework already returns them to the model
     with an instruction to fix them. A failure nobody in the conversation can fix, such as a courier
     API that is down: return an error that says not to retry, and hand over to a person. A bug in
     your own code: raise, and the run stops with no answer. A trap panel: returning an error saying
     out of stock is a lie, because the lookup worked.">

| What happened | Example | What the tool does | What the model sees |
|---|---|---|---|
| A fact about the world | `stock: 0`, no orders | returns it as data: the lookup worked | the fact |
| A request the model can fix | wrong order ID, ambiguous name | returns `{"error": ...}` saying what would work | the error, and a chance to retry |
| Bad arguments | `order` instead of `order_id` | nothing: the framework catches it | an error asking it to fix the call |
| Nobody in this conversation can fix it | courier API down after retries | returns an error that says so, and hand over | "don't retry; a colleague will follow up" |
| A bug in your code | `KeyError` | raises | nothing: the run stops |

**Docs:** [tool error handling](https://docs.langchain.com/oss/python/langchain/tools#error-handling)

### 8.1 A tool that raises

**Predict first:** a tool raises an exception inside `create_agent`. Does the agent catch it and tell
the model, or does the exception reach you?

In [27]:
@tool("get_shipment")
def get_shipment_down(order_id: str) -> dict:
    """Track a shipped order: carrier, tracking ID, status and estimated arrival date."""
    raise ConnectionError("courier API timed out after 3 attempts")


broken = create_agent(model, others + [get_shipment_down], system_prompt=v1.PROMPTS["agent_prompt"])
try:
    answer(broken, "Where is my parcel for ORD-5003?")
except Exception as err:
    print(type(err).__name__, ":", err)
    print("note:", err.__notes__)

ConnectionError : courier API timed out after 3 attempts
note: ["During task with name 'tools' and id '43542b5c-10f4-1635-85d9-0dd20cf3e09d'"]


It reaches you. The run is over: no partial answer, no apology, and the traceback points at the
framework's `tools` step rather than at your function. That's the right behaviour for a **bug**. A bug
should stop things loudly, and a model can't fix your `KeyError`. It's the wrong behaviour for a
*foreseeable* failure like a missing order, which is why ShopWise's tools return those.

### 8.2 Bad arguments are caught for you

Malformed arguments are different. To show one without waiting for a model to make the mistake, give
`create_agent` a fake model that replays a script (appendix E uses the same trick for testing):

In [28]:
from langchain_core.language_models.fake_chat_models import GenericFakeChatModel


class ScriptedModel(GenericFakeChatModel):
    """Replays the messages you give it. create_agent calls bind_tools, so accept and ignore it."""

    def bind_tools(self, tools, **kwargs):
        return self


bad_call = AIMessage(content="", tool_calls=[
    {"name": "get_order", "args": {"order": "ORD-5003"}, "id": "call_1", "type": "tool_call"}])
scripted = create_agent(ScriptedModel(messages=iter([bad_call, AIMessage("done")])), shop.TOOLS)
result = scripted.invoke({"messages": [{"role": "user", "content": "What's in ORD-5003?"}]})

tool_message = result["messages"][2]
print(tool_message.status, "|", tool_message.content)

error | Error invoking tool 'get_order' with kwargs {'order': 'ORD-5003'} with error:
 order_id: Field required
 Please fix the error and try again.


The arguments never reached your function. The schema check failed, and the framework handed the
model a `ToolMessage` with `status="error"` and an instruction to fix the call. Only **argument
validation** gets this treatment; anything your function raises is section 8.1.

### 8.3 An error message is a prompt

When a tool returns `{"error": ...}`, those words are all the model has to decide what to do next.
Same outage, two wordings, three runs each. Count how often the agent calls `get_shipment` per run:

In [29]:
@tool("get_shipment")
def shipment_try_later(order_id: str) -> dict:
    """Track a shipped order: carrier, tracking ID, status and estimated arrival date."""
    return {"error": "courier API timed out after 3 attempts. Try again later."}


@tool("get_shipment")
def shipment_dont_retry(order_id: str) -> dict:
    """Track a shipped order: carrier, tracking ID, status and estimated arrival date."""
    return {"error": "Tracking is unavailable right now and retrying won't help. "
                     "Tell the customer a colleague will email the tracking details."}


for label, version in [("'try again later'", shipment_try_later), ("'retrying won't help'", shipment_dont_retry)]:
    variant = create_agent(model, others + [version], system_prompt=v1.PROMPTS["agent_prompt"])
    calls = []
    for run in range(3):
        result = answer(variant, "Where is my parcel for ORD-5003?")
        calls.append(score_tools.tools_called(result).count("get_shipment"))
    print(f"{label:24} get_shipment calls per run: {calls}")
    print(f"{'':24} {result['messages'][-1].text[:110]}")

'try again later'        get_shipment calls per run: [2, 2, 2]
                         Your order ORD-5003 has shipped, but the courier tracking service is temporarily unavailable. Please try check


'retrying won't help'    get_shipment calls per run: [1, 1, 1]
                         I checked the system for order ORD-5003, but tracking is temporarily unavailable right now. A colleague will e


"Try again later" reads as an instruction, and the model took it literally: it called the dead tool
again within the same second, which helps nobody. "Retrying won't help" got one call and a straight
answer. An error should say what to do *instead*. Behaviour like this belongs to this model and this
wording; it isn't a guarantee, which is exactly why you measure it.

### 8.4 Where retries belong

Retrying a flaky backend is your code's job, not the model's: `max_retries` on an HTTP client, a
retry with backoff inside the tool, and only after those are spent, an error the model can explain to
the customer. Session 7 does this for every tool at once, with `ToolRetryMiddleware` and
`ToolErrorMiddleware` wrapped round the tool step you wrote by hand in section 4.3. And some failures
must stop the loop rather than be explained: anything involving money, and anything that means your
code is wrong.

**The cost of returning errors:** a caller that isn't a model (a script, a test, another function) has
to remember `if "error" in result`. Python won't remind it. That's the price of the contract, and it's
worth paying for a caller who reads English.

## 9. A typed object at the end

Sometimes the caller wants data, not prose: a dashboard, a router, the next step of a pipeline.
`response_format` makes the agent finish with a validated object, here session 3's `TriageResult`,
unchanged.

**Docs:** [structured output](https://docs.langchain.com/oss/python/langchain/structured-output) ·
[in create_agent](https://docs.langchain.com/oss/python/langchain/agents#structured-output)

In [30]:
from langchain.agents.structured_output import ProviderStrategy, ToolStrategy
from schemas import TriageResult

typed = create_agent(model, shop.TOOLS, system_prompt=v1.PROMPTS["agent_prompt"],
                     response_format=ToolStrategy(TriageResult))
result = answer(typed, "I'm tanvir@example.com. My smartwatch still hasn't arrived and the courier says in transit. How long?")

print(repr(result["structured_response"]), "\n")
print("tools asked for:", score_tools.tools_called(result))
print("last message:   ", type(result["messages"][-1]).__name__, "|", result["messages"][-1].content[:90])

TriageResult(category='shipping', urgency='low', sentiment='calm', needs_human=False, summary='Tanvir asks about the delivery ETA for their smartwatch order ORD-5003.') 

tools asked for: ['lookup_customer', 'list_orders', 'get_shipment', 'get_product', 'TriageResult']
last message:    ToolMessage | Returning structured response: category='shipping' urgency='low' sentiment='calm' needs_hu


The typed object is in `result["structured_response"]`, **not in the message list**. With
`ToolStrategy`, the schema is offered to the model as one more tool; calling it ends the run. The last
message is a `ToolMessage` whose content is the object's `repr()`: it looks exactly like the data and
it's a string. Read `structured_response`.

There's also no prose reply any more: the structured output *replaces* the final message. An agent
that has to both answer the customer and report a classification needs both, which is one reason v2
keeps `triage.py` as a separate entry point (ADR-0007). And notice the category, `shipping`. Triage's
convention for "where is it?" is `order_status`; the agent never saw those conventions, so its
classification isn't triage's, and isn't scored.

### 9.1 Name the strategy

`response_format=TriageResult` with no strategy lets LangChain choose: **provider-native** structured
output if the model's profile says it supports it, a tool call otherwise. The choice depends on a
dictionary you didn't write:

In [31]:
strong = init_chat_model(f"{settings.PROVIDER}:{settings.MODEL_STRONG}")
for name, m in [(settings.MODEL, model), (settings.MODEL_STRONG, strong)]:
    print(f"{name:24} profile says structured_output: {(m.profile or {}).get('structured_output')}")

stuck = create_agent(strong, shop.TOOLS, system_prompt=v1.PROMPTS["agent_prompt"],
                     response_format=ProviderStrategy(TriageResult))
try:
    stuck.invoke({"messages": [{"role": "user", "content": "I'm tanvir@example.com. Where is my smartwatch?"}]},
                 config={"recursion_limit": 10})
except Exception as err:
    print("\nProviderStrategy on", settings.MODEL_STRONG, "->", type(err).__name__)

gemini-3.5-flash-lite    profile says structured_output: True
gemini-3.8-flash         profile says structured_output: None


Measured while building this session:

| | `MODEL` (profile: supported) | `MODEL_STRONG` (no profile) |
|---|---|---|
| `ToolStrategy(TriageResult)` | works | works |
| `TriageResult` (auto) | picks provider-native, works | picks `ToolStrategy`, works |
| `ProviderStrategy(TriageResult)` | works | **never finishes**: loops until the budget stops it |

Nothing in that error mentions `response_format`. Without a budget it would have run up to 9,999 steps.
So: pass the strategy explicitly (`ToolStrategy` works on every model that can call tools), and keep
a budget on every run, which turns "hangs" into an error you can read.

## 10. Project v2

<img src="figures/v2-architecture.png" width="900"
     alt="The v2 architecture. A customer message enters create_agent, which holds the grounded agent
     prompt from prompts.yml. The agent sends tool requests to tools.py, six read-only tools, and
     gets results back. The tools read ShopWise's records through shopwise_data.py, a five-line
     loader, from one shared file, backend.yml. A seventh tool, issue_refund, sits beside the six
     behind a padlock: defined and working, not given to the agent until session 7. Out of the agent
     come a grounded reply or, with ToolStrategy, a typed TriageResult. Off to one side, triage.py
     remains the other entry point, and two scoring scripts sit beside the snapshot: score.py for
     triage and score_tools.py for tool selection.">

| File | What changed | Why it's separate |
|---|---|---|
| `shopwise_data.py` | **new**: loads `data/backend.yml` into five dicts | the seam where a real database would go |
| `tools.py` | **new**: seven tools, `TOOLS` holds the six that read | what the model may ask for, and nothing else |
| `agent.py` | **new**: `build_agent()`, `answer()`, `transcript()`, `MAX_STEPS` | the second entry point (ADR-0007) |
| `prompts.yml` | gains `agent_prompt` | wording lives with the other prompts |
| `config.py` | `VERSION` 2.0.0, `MODEL_PRO` → `gemini-3.1-pro-preview`; **no new setting** | the tools reach the data through `DATA_DIR`, there since v0.2 |
| `triage.py`, `utils.py`, `schemas.py`, `policy.py` | unchanged, byte for byte | triage is the other job |
| `../score.py`, `../score_tools.py` | `score_tools.py` is new | they measure the assistant; they aren't part of it |

`MODEL_PRO` moves because Google no longer opens the 2.5 models to new keys; it's a preview model and
needs a billed key, and nothing in sessions 5 to 7 uses it.

Walk through the files in your editor. In a terminal, from `session-05/project/`:

```bash
uv run python tools.py                                  # every tool, called directly: no key
uv run python agent.py                                  # three demo questions
uv run python agent.py "Is the kettle in stock?"        # your own
uv run python ../score_tools.py                         # tool selection, 15 questions
uv run python ../score.py                               # triage, 20 tickets
```

The same, run from here so the outputs are saved:

In [32]:
for command in [["tools.py"], ["agent.py"]]:
    run = subprocess.run([sys.executable, *command], cwd="project", capture_output=True, text=True)
    print(run.stdout or run.stderr[-800:])

given to the agent: ['lookup_customer', 'get_order', 'list_orders', 'get_shipment', 'get_product', 'check_refund_status']
withheld:           issue_refund 

{'email': 'tanvir@example.com', 'customer_id': 'C-1002', 'name': 'Tanvir Ahmed', 'tier': 'regular', 'since': '2024-07-02'}
[{'order_id': 'ORD-5003', 'status': 'shipped', 'order_date': '2026-07-28', 'total': 199.0}]
{'order_id': 'ORD-5003', 'customer_id': 'C-1002', 'items': [{'sku': 'SKU-200', 'name': 'PulseFit Smartwatch', 'qty': 1, 'unit_price': 199.0}], 'total': 199.0, 'status': 'shipped', 'order_date': '2026-07-28', 'delivered_date': None}
{'order_id': 'ORD-5003', 'carrier': 'SwiftEx', 'tracking_id': 'SW-88231', 'status': 'in_transit', 'eta': '2026-08-08'}
{'error': 'no tracking for ORD-5002. get_order shows its current status.'}
{'sku': 'SKU-300', 'name': 'BrewMaster Electric Kettle', 'product_type': 'home', 'price': 49.0, 'stock': 0, 'warranty_months': 6}
{'refund_id': 'REF-9001', 'order_id': 'ORD-5001', 'amount': 129.0, 'stat

ShopWise Support Assistant v2.0.0 · google_genai:gemini-3.5-flash-lite · 6 tools · budget 12 steps

CUSTOMER   Hi, I'm tanvir@example.com. What did I order most recently, and when will it arrive?
  ASKS     lookup_customer({'email': 'tanvir@example.com'})
  RESULT   lookup_customer -> {"email": "tanvir@example.com", "customer_id": "C-1002", "name": "Tanvir Ahmed", "tier": "regular", "since": "2024-07-02"}
  ASKS     list_orders({'customer_id': 'C-1002'})
  RESULT   list_orders -> [{"order_id": "ORD-5003", "status": "shipped", "order_date": "2026-07-28", "total": 199.0}]
  ASKS     get_shipment({'order_id': 'ORD-5003'})
  RESULT   get_shipment -> {"order_id": "ORD-5003", "carrier": "SwiftEx", "tracking_id": "SW-88231", "status": "in_transit", "eta": "2026-08-08"}
  ASKS     get_order({'order_id': 'ORD-5003'})
  RESULT   get_order -> {"order_id": "ORD-5003", "customer_id": "C-1002", "items": [{"sku": "SKU-200", "name": "PulseFit Smartwatch", "qty": 1, "unit_price": 199.0}], "total": 199.

### 10.1 The regression check

Triage wasn't supposed to move tonight. The proof is a byte comparison, not a score:

In [33]:
for name in ["triage.py", "utils.py", "schemas.py", "policy.py"]:
    same = (Path("project") / name).read_bytes() == (Path("../session-04/project") / name).read_bytes()
    print(f"{name:11} {'identical to v1' if same else 'CHANGED'}")

run = subprocess.run([sys.executable, "score.py"], capture_output=True, text=True)
print("\n" + run.stdout[-400:])

triage.py   identical to v1
utils.py    identical to v1
schemas.py  identical to v1
policy.py   identical to v1



scoring google_genai:gemini-3.5-flash-lite on 20 tickets, 1 run(s)

run 1: triage 18/20 · 1127 output tokens · order numbers 7 found, 0 missed, 0 invented, 0 rejected
    TCK-005  labelled return            model said warranty
    TCK-016  labelled refund            model said order_status



The code is identical, so any movement in the score is the model, not v2. These same files scored
17/20 in session 4's saved run, and 19/20 on another run while this session was being built. One run
is a sample: on twenty tickets, two or three either way is noise. `score.py --runs 3` shows the
spread; session 9 turns this into proper experiments.

## 11. Wrap-up

### 11.1 v1 → v2

| | v1: a brain | v2: with hands |
|---|---|---|
| An order's status | apologised for, or invented once the constraints came out | looked up, and stated only if a tool returned it |
| A question needing three facts | one call, one guess | three lookups chained, in an order nobody wrote |
| What "grounded" means | a paragraph asking the model to admit ignorance | a lookup, and an instruction to say only what came back |
| A tool that fails | there were no tools | a choice: return what the model can fix, raise what it can't |
| Tool quality | n/a | scored: 15 questions, the set of tools each must call |
| A typed result | `triage()` only | `structured_response` from an agent, with an explicit strategy |
| Entry points | one | two, deliberately: triage and the agent (ADR-0007) |

### 11.2 Where tools go next: MCP

The tool definition you've been reading all evening, a name, a description and a JSON schema for the
arguments, is exactly what an **MCP server** publishes. The Model Context Protocol is the industry's
standard way to offer tools to any agent: write `get_order` once as an MCP server, and Claude Code,
an IDE, or ShopWise's own agent can all call it. Session 19 builds one over this backend. Its tools
follow every rule in section 7, and its risks are section 3.4's: a tool you didn't write can do
anything its description doesn't mention. **Docs:** [MCP specification](https://modelcontextprotocol.io/specification/2026-07-28) ·
[MCP in LangChain](https://docs.langchain.com/oss/python/langchain/mcp)

### 11.3 Interview questions

1. *"So the model executes the function?"* Correct it precisely: the model emits a **request**, a name
   and arguments in the place its text would have been, and your code decides whether to run it.
   Because your code decides, it can refuse, which is how a human gets into the loop.
2. *Why are type hints load-bearing on a tool?* They aren't documentation: they build the schema the
   model fills in. Delete one and the model is free to send anything.
3. *Your agent answers from memory instead of calling the tool. Name three levers.* The tool's name
   and argument names (which section 7.2 shows carry most of the selection), its docstring, and the
   system prompt's grounding rule. Then measure, don't guess.
4. *A tool fails. Raise or return?* Return what the model can act on, with what would work; raise
   what it can't, so the run stops loudly.

### 11.4 Tanvir's follow-up

Two questions, a few seconds apart, as two separate requests, which is how every message from a web
chat arrives:

In [34]:
from agent import build_agent

v2 = build_agent()
print(transcript(answer(v2, "I'm tanvir@example.com. Where is my order ORD-5003?")), "\n")
print(transcript(answer(v2, "And is that the one with the smartwatch in it?")))

CUSTOMER   I'm tanvir@example.com. Where is my order ORD-5003?
  ASKS     lookup_customer({'email': 'tanvir@example.com'})
  RESULT   lookup_customer -> {"email": "tanvir@example.com", "customer_id": "C-1002", "name": "Tanvir Ahmed", "tier": "regular", "since": "2024-07-02"}
  ASKS     get_shipment({'order_id': 'ORD-5003'})
  RESULT   get_shipment -> {"order_id": "ORD-5003", "carrier": "SwiftEx", "tracking_id": "SW-88231", "status": "in_transit", "eta": "2026-08-08"}
ASSISTANT  Your order ORD-5003 is currently in transit with SwiftEx. The tracking ID is SW-88231, and it is expected to arrive by August 8, 2026. 



CUSTOMER   And is that the one with the smartwatch in it?
ASSISTANT  Please let me know your email address and order number so I can check which order contains your smartwatch.


The second request starts from an empty message list, so "that" refers to nothing. Here the agent
asked Tanvir for the email and order number he gave seconds earlier. On other runs while this session
was being built it looked the smartwatch up as a *product* instead, and answered a question he hadn't
asked (once quoting the price in a currency the record doesn't contain). There's nowhere for the
conversation to live: in section 4 the whole memory was a local variable called `messages`.

Now Tanvir asks about an order that isn't his:

In [35]:
print(transcript(answer(v2, "I'm tanvir@example.com. What was in order ORD-5001, and has it been delivered?")))

CUSTOMER   I'm tanvir@example.com. What was in order ORD-5001, and has it been delivered?
  ASKS     lookup_customer({'email': 'tanvir@example.com'})
  ASKS     get_order({'order_id': 'ORD-5001'})
  ASKS     get_shipment({'order_id': 'ORD-5001'})
  RESULT   lookup_customer -> {"email": "tanvir@example.com", "customer_id": "C-1002", "name": "Tanvir Ahmed", "tier": "regular", "since": "2024-07-02"}
  RESULT   get_order -> {"order_id": "ORD-5001", "customer_id": "C-1001", "items": [{"sku": "SKU-100", "name": "AeroSound Pro Headphones", "qty": 1, "unit_price": 129.0}], "total": 129.0, "status": "delivered", "order_date": "2026-07-14", "delivered_date": "2026-07-19"}
  RESULT   get_shipment -> {"error": "no tracking for ORD-5001. get_order shows its current status."}
ASSISTANT  Order ORD-5001 contained one AeroSound Pro Headphones. It was delivered on 19 July 2026.


Read the two results. `lookup_customer` says Tanvir is `C-1002`. `get_order` says `ORD-5001` belongs to
`C-1001`. Both facts are in the model's context, and it reads another customer's order out anyway,
accurately and politely. Nothing in this design checks who is asking; the model isn't an access
check, and a prompt rule asking it to be one would be a rule enforced some of the time.

**Grounding is not memory, and it isn't authorisation.** Session 6 gives the conversation somewhere to
live and a way to show progress while the tools run. Session 7 puts a check at the point where your
code runs the tool, the gap you wrote by hand in section 4.3.

**Before then:** `exercises.ipynb`. The ⭐⭐⭐ ones finish the hand-written loop and measure a tool
description of your own.

---

# Appendix

Optional, and not needed for session 6.

## A. Forcing or forbidding a tool call

`tool_choice` overrides the model's decision. `"any"` forces some tool, a tool's name forces that one,
`"none"` forbids tools for this call.

In [36]:
for choice in ["get_product", "any", "none"]:
    reply = model.bind_tools(shop.TOOLS, tool_choice=choice).invoke("Hello! Where is ORD-5003?")
    print(f"{choice:12} asks {[c['name'] for c in reply.tool_calls]}  {reply.text[:70]!r}")

get_product  asks ['get_product']  ''


any          asks ['get_shipment']  ''


none         asks []  'It looks like "ORD-5003" could refer to a few different things dependi'


Useful in a fixed pipeline where the next step must be a lookup, or to force a final answer with no
more calls. Don't force a tool inside an agent loop: the model can then never stop asking.
**Docs:** [forcing tool calls](https://docs.langchain.com/oss/python/langchain/models#forcing-tool-calls)

## B. `return_direct`, and when it's wrong

`@tool(return_direct=True)` ends the run as soon as that tool returns, and the tool's output becomes
the answer. That's right when the output *is* the answer, a formatted receipt for example. It's wrong
far more often, always for the same reason: the model no longer gets to use the result. Every
multi-hop run tonight would have stopped after `lookup_customer`.

## C. Structured output: the other errors

| Error | Means | Handled by default under `ToolStrategy`? |
|---|---|---|
| `StructuredOutputValidationError` | the object failed your schema's validators (session 3's *legal but not right*) | yes: the error is sent back and the model retries |
| `MultipleStructuredOutputsError` | the model filled the schema twice in one message | yes, same |

`ToolStrategy(TriageResult, handle_errors=...)` controls that retry: `True` (the default), a fixed
message, or a function that writes one. It's section 8's return-don't-raise applied to the schema.

## D. Reasoning where it changes the action

Session 2 compared a hand-written chain of thought with `thinking_level`. In an agent, reasoning can
change *which tool is called*, so measure it on the tool-selection questions:

In [37]:
thinker = init_chat_model(settings.model_id, thinking_level="high")
for label, m in [("default (minimal)", model), ("thinking_level=high", thinker)]:
    variant = create_agent(m, shop.TOOLS, system_prompt=v1.PROMPTS["agent_prompt"])
    start = time.perf_counter()
    s = score_tools.score(variant, score_tools.load_questions(), show=False)
    print(f"{label:22} {s['passed']}/{s['total']}  {time.perf_counter() - start:5.1f} s  "
          f"{s['model_calls']} model calls")

default (minimal)      15/15   27.2 s  34 model calls


thinking_level=high    15/15   42.9 s  33 model calls


On easy tool choices, extra reasoning buys nothing it can show and costs latency on every call. It
earns its cost when the choice is genuinely hard: many similar tools, or a plan that has to be right
before the first call. That's a measurement to repeat on your own tools, not a rule.

## E. Testing an agent without a model

Section 8.2's `ScriptedModel` replays a fixed conversation, so you can test **your** side of the loop
(tools, wiring, transcript handling) with no key and no randomness:

In [38]:
def test_lookup_chain():
    script = iter([
        AIMessage("", tool_calls=[{"name": "lookup_customer", "args": {"email": "tanvir@example.com"},
                                   "id": "1", "type": "tool_call"}]),
        AIMessage("", tool_calls=[{"name": "list_orders", "args": {"customer_id": "C-1002"},
                                   "id": "2", "type": "tool_call"}]),
        AIMessage("Your PulseFit Smartwatch is on its way."),
    ])
    agent = create_agent(ScriptedModel(messages=script), shop.TOOLS)
    result = agent.invoke({"messages": [{"role": "user", "content": "anything"}]})

    tool_results = [m.content for m in result["messages"] if m.type == "tool"]
    assert '"customer_id": "C-1002"' in tool_results[0]
    assert "ORD-5003" in tool_results[1]
    assert result["messages"][-1].text.startswith("Your PulseFit")


test_lookup_chain()
print("passed")

passed


This tests the tools and the plumbing, not the model's judgement; section 7.1's score does that.
`GenericFakeChatModel` has no `bind_tools` of its own, hence the three-line subclass (the docs' own
example passes `tools=[]` to avoid it). Session 13 moves tests like this into `pytest`.
**Docs:** [unit testing agents](https://docs.langchain.com/oss/python/langchain/test/unit-testing)

## F. `create_agent`'s other arguments

| Argument | Does | Session |
|---|---|---|
| `response_format=` | a typed object at the end | tonight, section 9 |
| `checkpointer=` | saves the message list between calls: memory | 6 |
| `middleware=` | code around every model and tool call: budgets, retries, approval | 7 |
| `context_schema=` | per-request facts for tools, e.g. who is logged in | 11 |
| `store=` | memory across conversations | 11 |
| `name=` | names the agent when it becomes one of several | 12 |